# Trabajo Práctico Integrador - Análisis de Datos


## 1. Depuración de datos

**Objetivo:** dejar el dataset *Telco Customer Churn* listo para el análisis: detectar y resolver los espacios vacíos de `TotalCharges`, corregir los tipos de datos (`dtypes`), tratar los valores nulos y exportar el resultado a `data/processed/`.

**Criterio general:** la limpieza es el paso previo al análisis. El tratamiento de valores perdidos es *una decisión metodológica del analista*, por lo que cada acción de esta sección queda justificada.

> El archivo crudo de `data/raw/` **no se modifica nunca**: toda la depuración se hace sobre una copia en memoria y se guarda como un archivo nuevo.

### 1.1 Importación de librerías

Se usa **Pandas** (lectura, inspección y transformación de datos tabulares) y **`os`** (módulo estándar de Python, para manejar rutas y carpetas).

In [ ]:
import os
import pandas as pd

### 1.2 Carga del dataset crudo

`pd.read_csv()` convierte el CSV en un `DataFrame`. 

**Reproducibilidad:** el notebook se ejecuta en la ruta local(`data/raw/`) y, si no existe, se lee el mismo archivo desde GitHub. Se parte siempre del mismo dato original.

In [ ]:
RUTA_RAW_LOCAL = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
URL_RAW = ("https://raw.githubusercontent.com/Janegro09/tpi-analisis-de-datos/"
           "main/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

ruta = RUTA_RAW_LOCAL if os.path.exists(RUTA_RAW_LOCAL) else URL_RAW
print("Leyendo desde:", ruta)

df = pd.read_csv(ruta)
filas_originales = len(df)  # se guarda para validar al final que no se perdieron filas

### 1.3 Comprensión inicial: estructura y tipos de datos

El primer paso es conocer la "forma" de los datos antes de tocarlos:

- `df.shape` → cantidad de (filas, columnas).
- `df.head()` → primeras filas, para ver cómo luce cada columna.
- `df.dtypes` / `df.info()` → tipo de dato y cantidad de valores no nulos por columna. Es la forma de detectar una columna que *conceptualmente* es numérica pero está guardada como texto.

In [ ]:
print("Filas y columnas:", df.shape)
df.head()

In [ ]:
df.info()

**Conclusión:** `TotalCharges` figura como `object`/`str` (texto) aunque es una variable **cuantitativa continua** (cargos acumulados). Esto contradice la clasificación conceptual de la fundamentación de negocio y es la señal de que hay algún valor no numérico cargado en esa columna. Además, `df.info()` muestra 0 nulos en todas las columnas: el problema está "escondido".

### 1.4 Detección de los espacios vacíos en `TotalCharges`

`df.isnull()` **no** detecta este problema, porque las celdas no son `NaN`: contienen un string con un espacio (`" "`). Pandas los trata como texto válido.

La detección se hace con:
- `.str.strip()` → elimina espacios al inicio y al final de cada texto, de modo que `" "` queda como `""`.
- `== ""` → genera una **máscara booleana** (`True` donde la celda quedó vacía).
- `.sum()` → cuenta los `True`.
- `df[mascara]` → filtra las filas problemáticas para inspeccionarlas.

In [1]:
mascara_vacios = df["TotalCharges"].str.strip() == ""

print("Celdas vacías en TotalCharges:", mascara_vacios.sum())
df.loc[mascara_vacios, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

NameError: name 'df' is not defined

**Análisis del patrón de no respuesta** (apunte de la cátedra: "patrones de la no respuesta"):

Las 11 filas tienen `tenure = 0`, es decir, son clientes recién dados de alta que **todavía no recibieron ninguna factura**. Por lo tanto el vacío **no es completamente aleatorio**: está explicado por otra variable del dataset. Esto condiciona cómo se trata (ver 1.6).

Se verifica que la relación sea total y no una coincidencia:

In [ ]:
# ¿Todos los vacíos tienen tenure = 0? ¿Y todos los clientes con tenure = 0 tienen vacío?
print("Vacíos con tenure == 0:     ", (df.loc[mascara_vacios, "tenure"] == 0).sum(), "de", mascara_vacios.sum())
print("Clientes con tenure == 0:   ", (df["tenure"] == 0).sum())

### 1.5 Conversión de tipos de datos (`dtypes`)

`pd.to_numeric(serie, errors="coerce")` convierte el texto a número y, cuando un valor **no se puede convertir** (como `" "`), lo reemplaza por `NaN` en lugar de lanzar un error. Es la misma técnica que se vio en la clase de limpieza de datos (ejemplo de `edad` y `salario`).

Ventajas:
- La columna pasa a `float64`, necesario para `describe()`, correlaciones y gráficos.
- Los vacíos se transforman en `NaN`, que Pandas sí reconoce como dato faltante y permite tratar con `isnull()`, `fillna()` y `dropna()`.

**Control de seguridad:** `errors="coerce"` convertiría en `NaN` *cualquier* valor inválido sin avisar. Por eso se verifica que la cantidad de `NaN` generados coincida exactamente con los 11 vacíos detectados: así se confirma que no había otros valores corruptos ocultos.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Nuevo dtype de TotalCharges:", df["TotalCharges"].dtype)
print("NaN generados por la conversión:", df["TotalCharges"].isnull().sum())

# Si falla, hay valores inválidos distintos de los espacios en blanco
assert df["TotalCharges"].isnull().sum() == mascara_vacios.sum()

### 1.6 Tratamiento de valores nulos

Ahora que los vacíos son `NaN`, se cuantifican con `isnull().sum()` (cantidad) y `isnull().mean() * 100` (porcentaje), como indica el apunte de EDA.

In [ ]:
nulos = df.isnull().sum()
pct_nulos = df.isnull().mean() * 100

pd.DataFrame({"nulos": nulos, "% nulos": pct_nulos.round(2)}).query("nulos > 0")

#### Decisión: imputar con `0` (`fillna(0)`)

Del apunte de la cátedra: *"el tratamiento de valores perdidos es una decisión metodológica del analista"*. Opciones evaluadas:

| Opción | Evaluación |
|---|---|
| `dropna()` (eliminar filas) | Sirve cuando la no respuesta es **completamente aleatoria**. No es el caso: los 11 nulos están explicados por `tenure = 0`. Además se perderían clientes reales (aunque sean solo el 0,16 %). |
| Imputar con media/mediana | **Incorrecto conceptualmente:** asignaría a clientes con cero meses de antigüedad un acumulado de ~1.400–2.300, un monto que nunca pagaron. Distorsionaría las correlaciones con `tenure`. |
| **Imputar con `0`** | **Elegida.** Es la imputación "con valor definido por criterio" del apunte. Tiene sentido de negocio: con 0 meses de antigüedad, el cargo total acumulado es exactamente 0. Conserva las 7043 filas y es coherente con `tenure`. |

> **Alcance:** el `0` se aplica **solo** a `TotalCharges`, que es donde está el problema. Se hace sobre la columna y no sobre todo el `DataFrame` para no alterar otras variables.

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Nulos restantes en TotalCharges:", df["TotalCharges"].isnull().sum())

# Las 11 filas imputadas ahora son coherentes con su tenure (0 meses -> 0 acumulado)
df.loc[mascara_vacios, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

### 1.7 Detección de duplicados

Los duplicados sobrerrepresentan registros y sesgan estimaciones (apunte "Data cleaning"). Se revisan dos niveles:

- `df.duplicated().sum()` → filas **completamente** repetidas.
- `df["customerID"].duplicated().sum()` → el identificador del cliente debería ser único (cada fila es un cliente).

Si hubiera duplicados se eliminarían con `drop_duplicates()`.

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())
print("customerID duplicados:", df["customerID"].duplicated().sum())

df = df.drop_duplicates()  # sin efecto si no hay duplicados; deja el paso documentado y reproducible

### 1.8 Revisión del resto de los tipos y categorías

- `SeniorCitizen` es `int64` con valores 0/1. Conceptualmente es categórica binaria, pero se **mantiene** como 0/1 para no alterar el dataset ni limitar su uso en la matriz de correlación posterior.
- `Churn` (variable objetivo) se **mantiene** como `Yes`/`No`, tal como está en el original. Su versión numérica se agrega aparte en 1.9.
- Se verifica con `unique()` que las columnas categóricas no tengan variantes escritas distintas (por ejemplo `"yes"`, `"Yes "`), que serían valores inconsistentes.

In [ ]:
columnas_categoricas = df.select_dtypes(exclude="number").columns.drop("customerID")

for col in columnas_categoricas:
    print(f"{col:18s}", sorted(df[col].unique()))

Las categorías son consistentes. Los valores `"No phone service"` y `"No internet service"` **no son errores**: son categorías válidas del negocio (el cliente no tiene ese servicio), por lo que no se modifican.

### 1.9 Codificación numérica de la variable objetivo (`Churn_num`)

`Churn` es la variable objetivo del estudio, pero está guardada como texto (`Yes`/`No`). Los métodos numéricos como `df.corr()` y el heatmap de correlación **solo trabajan con columnas numéricas**, así que con `Churn` como texto el target quedaría fuera de la matriz.

Se crea una **columna nueva** `Churn_num` con `Series.map()`, que reemplaza cada valor según un diccionario: `No` → 0 (el cliente continúa) y `Yes` → 1 (el cliente abandonó).

- La columna original `Churn` **no se modifica**: `Churn_num` es una copia traducida, no se pierde ni se inventa información.
- Con esta codificación binaria, el promedio de `Churn_num` equivale directamente a la **tasa de abandono**.
- `map()` devuelve `NaN` para cualquier valor que no esté en el diccionario; por eso se verifica que no haya nulos y que ambas columnas coincidan.

In [ ]:
df["Churn_num"] = df["Churn"].map({"No": 0, "Yes": 1})

# Control: el mapeo no dejó nulos y es equivalente a la columna original
assert df["Churn_num"].isnull().sum() == 0, "Hay valores de Churn fuera de Yes/No"

print(pd.crosstab(df["Churn"], df["Churn_num"]))
print(f"Tasa de abandono: {df['Churn_num'].mean():.1%}")

### 1.10 Validación final

Antes de exportar se comprueba con `assert` que se cumple todo lo esperado. Si algo falla, el notebook se detiene en lugar de exportar un dataset incorrecto.

In [ ]:
assert len(df) == filas_originales, "Se perdieron filas durante la depuración"
assert df.isnull().sum().sum() == 0, "Quedan valores nulos"
assert df["TotalCharges"].dtype == "float64", "TotalCharges no es numérica"
assert df["customerID"].is_unique, "Hay clientes duplicados"
assert df.shape[1] == 22, "Se esperaban 21 columnas originales + Churn_num"
assert (df["Churn"].eq("Yes") == df["Churn_num"].eq(1)).all(), "Churn y Churn_num no coinciden"

print("Validación OK:", df.shape)
df.info()

In [ ]:
df[["tenure", "MonthlyCharges", "TotalCharges"]].describe()

### 1.11 Exportación a `data/processed/`

`df.to_csv()` guarda el DataFrame depurado en un archivo nuevo:
- `index=False` evita escribir el índice de Pandas como una columna extra.
- `os.makedirs(..., exist_ok=True)` crea la carpeta de destino si no existe (por ejemplo en Colab) sin fallar si ya existe.

> En Google Colab el archivo se guarda en el disco temporal de la sesión. Para versionarlo hay que descargarlo y subirlo a `data/processed/` del repositorio.

In [ ]:
RUTA_PROCESSED = "../data/processed"
ARCHIVO_LIMPIO = os.path.join(RUTA_PROCESSED, "telco_churn_clean.csv")

os.makedirs(RUTA_PROCESSED, exist_ok=True)
df.to_csv(ARCHIVO_LIMPIO, index=False)

print("Dataset depurado guardado en:", ARCHIVO_LIMPIO)

**Resumen de la depuración**

| Acción | Resultado |
|---|---|
| Detección de espacios vacíos en `TotalCharges` | 11 celdas (todas con `tenure = 0`) |
| Conversión de tipo | `TotalCharges`: `object` → `float64` |
| Tratamiento de nulos | Imputación con `0` (cliente sin facturación aún) |
| Duplicados | 0 encontrados |
| Codificación del target | Nueva columna `Churn_num` (`No` → 0, `Yes` → 1); `Churn` original intacta |
| Filas / columnas finales | 7043 × 22 (21 originales + `Churn_num`, sin pérdida de registros) |
| Archivo exportado | `data/processed/telco_churn_clean.csv` |